In [ ]:
%useLatestDescriptors
%use serialization, kandy

In [ ]:
@Serializable
public data class JmhReport(
    val jmhVersion: String,
    val benchmark: String,
    val mode: String,
    val threads: UInt,
    val forks: UInt,
    val jvm: String,
    val jvmArgs: List<String>,
    val jdkVersion: String,
    val vmName: String,
    val vmVersion: String,
    val warmupIterations: UInt,
    val warmupTime: String,
    val warmupBatchSize: UInt,
    val measurementIterations: UInt,
    val measurementTime: String,
    val measurementBatchSize: UInt,
    val params: Map<String, String> = emptyMap(),
    val primaryMetric: PrimaryMetric,
    val secondaryMetrics: Map<String, SecondaryMetric>,
) {
    public interface Metric {
        public val score: Double
        public val scoreError: Double
        public val scoreConfidence: List<Double>
        public val scorePercentiles: Map<Double, Double>
        public val scoreUnit: String
    }

    @Serializable
    public data class PrimaryMetric(
        override val score: Double,
        override val scoreError: Double,
        override val scoreConfidence: List<Double>,
        override val scorePercentiles: Map<Double, Double>,
        override val scoreUnit: String,
        val rawDataHistogram: List<List<List<List<Double>>>>? = null,
        val rawData: List<List<Double>>? = null,
    ) : Metric

    @Serializable
    public data class SecondaryMetric(
        override val score: Double,
        override val scoreError: Double,
        override val scoreConfidence: List<Double>,
        override val scorePercentiles: Map<Double, Double>,
        override val scoreUnit: String,
        val rawData: List<List<Double>>,
    ) : Metric
}

In [ ]:
import java.io.File

@OptIn(ExperimentalSerializationApi::class)
val reports = Json.decodeFromStream<List<JmhReport>>(File("data/arrayAllocation-1.json").inputStream())

In [ ]:
val reportsByName = reports.groupBy { it.benchmark }.mapValues { it.value.sortedBy { it.params["size"]!!.toInt() } }
val dataByName = reportsByName.mapValues {
    val value = it.value
    mapOf(
//        "size" to value.map { it.params["size"]!!.toInt().toDouble() },
        "size" to value.indices.map { it.toDouble() },
        "score" to value.map { it.primaryMetric.score },
        "error" to value.map { it.primaryMetric.scoreError },
    )
}

val plots = dataByName.mapValues { (name, data) ->
    val xs = data["size"]!!
    val scores = data["score"]!!
    val errors = data["error"]!!
    val yMins = scores.zip(errors) { score, error -> log2(score - error) }
    val yMaxs = scores.zip(errors) { score, error -> log2(score + error) }
    val minShift = yMins.zip(xs) { y, x -> y - x }.min()
    val maxShift = yMaxs.zip(xs) { y, x -> y - x }.max()
    plot(data) {
        layout.title = name.substringAfter("dev.lounres.kone.benchmarks.collections.")
        x(xs, name = "log2(size)")
        y.axis.name = "log2(time)"
        errorBars {
            yMin(yMins)
            yMax(yMaxs)
        }
        line {
            y(xs.map { it + minShift })
        }
        line {
            y(xs.map { it + maxShift })
        }
    }
}

plotGrid(plots.values.toList(), nCol = 2)
//reportsByName.mapValues { it.value.last().primaryMetric.score }.values.max() / 10.0.pow(9)

In [ ]:
val start = 2

val data = buildList {
    addAll(1 ..< (1 shl start))
    for (i in start .. 29) {
        addAll((1 shl i) ..< (1 shl (i + 1)) step (1 shl (i - start)))
    }
}

println(data.size)

plot {
    x(data.map { log2(it.toDouble()) })
    points {
        y(data.map { log2(it.toDouble()) })
    }
}